# Probability Distributions — Node 23

This notebook is a hands-on companion to `README.md` and `lesson.tex`.

We will:
1. Define and tabulate the **PMF** of a discrete distribution (Binomial).
2. Estimate it empirically from Monte Carlo samples and compare.
3. Construct the **CDF** of a continuous distribution (Uniform(0,1)) both analytically and empirically.
4. Sanity-check the bijection theorem (CDF determines distribution) numerically.

Stdlib only — `math` and `random`.


In [ ]:
import math
import random

random.seed(0)


## 1. Binomial(5, 0.3) PMF — analytic

The PMF is
$$p_X(k) = \binom{n}{k} p^k (1-p)^{n-k}, \qquad k = 0, 1, \ldots, n.$$


In [ ]:
n, p = 5, 0.3

def binom_pmf(n, p, k):
    return math.comb(n, k) * p**k * (1 - p)**(n - k)

analytic = {k: binom_pmf(n, p, k) for k in range(n + 1)}
for k, v in analytic.items():
    print(f"p_X({k}) = {v:.5f}")
print(f"sum     = {sum(analytic.values()):.5f}")


## 2. Empirical PMF from 10,000 samples

Each Binomial draw is built from $n$ Bernoulli$(p)$ trials.


In [ ]:
N = 10_000

def sample_binomial(n, p):
    return sum(1 for _ in range(n) if random.random() < p)

samples = [sample_binomial(n, p) for _ in range(N)]
empirical = {k: samples.count(k) / N for k in range(n + 1)}

print(f"{'k':>3} {'analytic':>10} {'empirical':>11} {'|diff|':>8}")
for k in range(n + 1):
    a, e = analytic[k], empirical[k]
    print(f"{k:>3} {a:>10.5f} {e:>11.5f} {abs(a - e):>8.5f}")


Agreement to ~1% is expected for $N = 10{,}000$. Standard error per cell is $\sqrt{p_k(1-p_k)/N} \approx 0.005$ in the worst case.

## 3. Uniform(0, 1) CDF — analytic vs. empirical

The CDF of $X \sim \text{Uniform}(0, 1)$ is $F_X(x) = x$ on $[0,1]$.


In [ ]:
M = 2000
u_samples = sorted(random.random() for _ in range(M))

def ecdf(samples, x):
    lo, hi = 0, len(samples)
    while lo < hi:
        mid = (lo + hi) // 2
        if samples[mid] <= x:
            lo = mid + 1
        else:
            hi = mid
    return lo / len(samples)

print(f"{'x':>5} {'F_true':>8} {'F_emp':>8} {'|diff|':>8}")
for i in range(11):
    x = i / 10
    print(f"{x:>5.2f} {x:>8.3f} {ecdf(u_samples, x):>8.3f} {abs(x - ecdf(u_samples, x)):>8.3f}")


The empirical CDF tracks $F(x) = x$ within $O(1/\sqrt{M})$ — the Glivenko–Cantelli theorem in action.

## 4. CDF bijection sanity check

If two distributions have identical CDFs, they are the same distribution. We compare two independent samples from Uniform(0,1) and confirm their empirical CDFs agree.


In [ ]:
sample_A = sorted(random.random() for _ in range(M))
sample_B = sorted(random.random() for _ in range(M))

max_gap = max(abs(ecdf(sample_A, x/100) - ecdf(sample_B, x/100)) for x in range(101))
print(f"sup |F_A - F_B| over a 0.01 grid: {max_gap:.4f}")
print("Two-sample Kolmogorov-Smirnov-style gap; -> 0 as M -> infty.")


## 5. Takeaways

- A **PMF** is a probability; a **PDF** is a density (can exceed 1). The **CDF** is the unifying object.
- Empirical PMFs / CDFs converge to their analytic counterparts at the $1/\sqrt{N}$ rate.
- In ML papers, "$z \sim \mathcal{N}(0, I)$" or "$y \sim \text{softmax}(\ell)$" are just invocations of named distributions defined exactly this way.
